# ED Agent Mesh — Flow Bundle v1.2 (Day 7)
**Default assay:** **Abbott ARCHITECT hs‑cTnI** (ng/L) for the 0/1h Δ classifier.  
(You can switch by editing `ASSAYS` or passing a different `assay_key`.)

> Clinical safety: thresholds sourced from ESC/manufacturer summaries; **verify with your lab**. All meds remain **assist‑only**.


In [ ]:

# --- Stage 1: Pandas guardrails + minimal schema ---
import pandas as pd
pd.options.mode.chained_assignment = 'raise'

import json, hashlib, time, os, yaml
from typing import Dict, Any, List, Tuple
from collections import defaultdict

ALLOWED_ACTIONS = {
    "order_ecg","request_labs","order_ct","request_vbga","request_abga",
    "ecg_interpret","ecg_alert","ecg_repeat","request_troponin_repeat",
    "page_team",
    "cap.score.crb65","cap.abx.duration","order.blood_cultures","order.urine_antigen","order.sputum",
    "rush.step",
    "bed.request","ed_hold","icu_downgrade"
}

PARAM_SCHEMAS: Dict[str, Dict[str, str]] = {
    "order_ecg": {"priority":"enum:STAT|ROUTINE"},
    "request_labs": {"panel_id":"str","priority":"enum:STAT|ROUTINE"},
    "order_ct": {"protocol":"str","priority":"enum:STAT|ROUTINE"},
    "request_vbga": {"site":"enum:venous"},
    "request_abga": {"site":"enum:arterial"},
    "ecg_interpret": {"ecg_id":"str","source":"enum:triage|ed|ems"},
    "ecg_alert": {"severity":"enum:CRITICAL|URGENT","phenotype":"str","confidence":"float"},
    "ecg_repeat": {"minutes":"int"},
    "request_troponin_repeat": {"minutes":"int","assay":"enum:hsTnT|hsTnI|unknown"},
    "page_team": {"team":"str","priority":"enum:STAT|URGENT|ROUTINE","reason":"str"},
    "cap.score.crb65": {"criteria":"list"},
    "cap.abx.duration": {"mild_mod_days":"int","severe_days":"int","stability_required_days":"int"},
    "order.blood_cultures": {"pairs":"int","timing":"str"},
    "order.urine_antigen": {"targets":"list"},
    "order.sputum": {"gram_and_culture_within_h":"int"}
}

MUST_REQUIRE_PERMIT = {"order_ct","request_labs"}

def _ensure_type(name: str, val: Any, want: str):
    if want == "str" and not isinstance(val, str): raise ValueError(f"param.{name} must be str")
    if want == "int" and not isinstance(val, int): raise ValueError(f"param.{name} must be int")
    if want == "float" and not isinstance(val, (int, float)): raise ValueError(f"param.{name} must be float")
    if want == "list" and not isinstance(val, list): raise ValueError(f"param.{name} must be list")

def _ensure_enum(name: str, val: Any, options: List[str]):
    if not isinstance(val, str) or val not in options: raise ValueError(f"param.{name} must be one of {options}")

def _validate_params(action: str, params: Dict[str, Any]) -> Dict[str, Any]:
    schema = PARAM_SCHEMAS.get(action)
    if not schema: return params or {}
    extra = set(params.keys()) - set(schema.keys())
    if extra: raise ValueError(f"unexpected params for {action}: {sorted(extra)}")
    missing = [k for k in schema.keys() if k not in params]
    if missing: raise ValueError(f"missing params for {action}: {missing}")
    for k, rule in schema.items():
        if rule.startswith("enum:"): _ensure_enum(k, params[k], rule.split(":",1)[1].split("|"))
        else: _ensure_type(k, params[k], rule)
    return params

def parse_and_validate_proposal(raw: str, encounter_id: str) -> Dict[str, Any]:
    d = json.loads(raw)
    for key in ("action","params","requires_permit","justification"):
        if key not in d: raise ValueError(f"missing required field: {key}")
    if d["action"] not in ALLOWED_ACTIONS: raise ValueError(f"action '{d['action']}' not allowed")
    d["params"] = _validate_params(d["action"], d["params"])
    if d["action"] in MUST_REQUIRE_PERMIT and d["requires_permit"] is not True:
        raise ValueError(f"action '{d['action']}' must require permit")
    return d

print("[Stage 1 ready]")


In [ ]:

# --- Policy Layer ---
from typing import Callable, Dict, Any, Tuple

def log_event(kind: str, action: str = "", note: str = "") -> None:
    print(f"{kind:<6} | {action} | {note}")

CANON = {
    "proposal.order.ecg":"order_ecg",
    "proposal.order.labs":"request_labs",
    "proposal.order.ct":"order_ct",
    "proposal.request.vbga":"request_vbga",
    "proposal.request.abga":"request_abga",
    "proposal.ecg.interpret":"ecg_interpret",
    "proposal.ecg.alert":"ecg_alert",
    "proposal.ecg.repeat":"ecg_repeat",
    "proposal.request.troponin_repeat":"request_troponin_repeat",
    "proposal.page_team":"page_team",
    "proposal.cap.score.crb65":"cap.score.crb65",
    "proposal.cap.abx.duration":"cap.abx.duration",
    "proposal.order.blood_cultures":"order.blood_cultures",
    "proposal.order.urine_antigen":"order.urine_antigen",
    "proposal.order.sputum":"order.sputum",
    "proposal.rush.step":"rush.step",
    "proposal.bed.request":"bed.request",
    "proposal.ed_hold":"ed_hold",
    "proposal.icu_downgrade":"icu_downgrade"
}

COOLDOWN_SEC = {
    "bed.request":   30*60,
    "ed_hold":       10*60,
    "order_ct":      365*24*3600,
    "order_ecg":     2*3600,
    "request_labs":  2*3600,
    "icu_downgrade": 60*60,
    "request_vbga":  60*60,
    "request_abga":  60*60,
    "ecg_alert":     30*60,
    "ecg_repeat":    15*60,
    "request_troponin_repeat": 60*60
}

REPEAT_POLICY = {
    "order_ct":     {"allow": False},
    "order_ecg":    {"allow": True, "interval_sec": 2*3600},
    "request_labs": {"allow": False},
    "request_vbga": {"allow": True, "interval_sec": 60*60},
    "request_abga": {"allow": True, "interval_sec": 60*60},
    "ecg_repeat":   {"allow": True, "interval_sec": 15*60},
    "request_troponin_repeat": {"allow": True, "interval_sec": 60*60}
}

from collections import defaultdict
_last_fired = defaultdict(float)
_completed = set()
_last_done  = {}
awaiting_capacity = defaultdict(bool)

_ecg_last_severity = {}

import json, hashlib, time

def _idem_key(enc: str, canon: str, params: Dict[str, Any]) -> str:
    blob = json.dumps({"e":enc,"a":canon,"p":params}, sort_keys=True)
    return hashlib.sha256(blob.encode()).hexdigest()

def _params_sig(params: Dict[str, Any]) -> str:
    return hashlib.sha1(json.dumps(params, sort_keys=True).encode()).hexdigest()

def on_capacity_change(encounter_id: str, icu_free_beds: int) -> None:
    awaiting_capacity[encounter_id] = (icu_free_beds == 0)

def _ecg_upgrade_bypass(enc, canon, params):
    if canon != "ecg_alert": return False
    new = params.get("severity","")
    prev = _ecg_last_severity.get(enc)
    _ecg_last_severity[enc] = new
    order = {"":0,"URGENT":1,"CRITICAL":2}
    return prev is not None and order.get(new,0) > order.get(prev,0)

def _can_emit(enc: str, canon: str, params: Dict[str, Any], now=None) -> Tuple[bool,str,str]:
    now = now or time.time()
    key = _idem_key(enc, canon, params)
    if key in _completed: return False, "already completed", key
    cd = COOLDOWN_SEC.get(canon, 0)
    if now - _last_fired[key] < cd:
        if _ecg_upgrade_bypass(enc, canon, params):
            return True, "cooldown bypass (upgrade)", key
        return False, f"cooldown {int(cd - (now - _last_fired[key]))}s", key
    if canon == "bed.request" and awaiting_capacity.get(enc, False):
        return False, "awaiting capacity change", key
    return True, "ok", key

def _can_repeat(enc: str, canon: str, params: Dict[str, Any], ctx: Dict[str, Any], now=None):
    now = now or time.time()
    rule = REPEAT_POLICY.get(canon, {"allow": False})
    sig  = _params_sig(params)
    rk   = (enc, canon, sig)
    if canon == "order_ct" and ctx.get("new_indication", False):
        return True, "new indication"
    if not rule.get("allow", False):
        return (rk not in _last_done), "repeat not allowed"
    interval = rule.get("interval_sec")
    last = _last_done.get(rk, 0)
    if (now - last) < (interval or 0):
        return False, f"repeat cooldown {int((interval or 0) - (now - last))}s"
    return True, "ok"

def policy_emit(encounter_id: str, action: str, params: Dict[str, Any],
                ctx: Dict[str, Any], do_emit):
    canon = CANON.get(action, action)
    if canon == "bed.request" and ctx.get("icu_free_beds", 0) == 0:
        awaiting_capacity[encounter_id] = True
    ok1, reason = _can_repeat(encounter_id, canon, params, ctx)
    ok2, why2, key = _can_emit(encounter_id, canon, params)
    if not ok1: log_event("block", action, reason); return False
    if not ok2: log_event("block", action, why2);   return False
    do_emit(action, params)
    _last_fired[key] = time.time()
    log_event("action", action, "emitted")
    return True

def policy_complete(encounter_id: str, action: str, params: Dict[str, Any]) -> None:
    canon = CANON.get(action, action)
    key   = _idem_key(encounter_id, canon, params)
    _completed.add(key)
    _last_done[(encounter_id, canon, _params_sig(params))] = time.time()
    log_event("audit", action, "completed")

print("[Policy ready]")


In [ ]:

# --- Event Bus + Capacity ---
from collections import deque
class EventBus:
    def __init__(self):
        self.queue = deque()
        self.log = []
    def propose(self, action, params):
        self.queue.append((action, params))
        self.log.append(("propose", action, params))
        log_event("emit", action, str(params))
eventbus = EventBus()

def raw_emit(action, params):
    eventbus.propose(action, params)

class Capacity:
    def __init__(self, icu_total=2, icu_occupied=2):
        self.icu_total = icu_total
        self.icu_occupied = icu_occupied
    @property
    def icu_free(self):
        return max(0, self.icu_total - self.icu_occupied)

capacity = Capacity(icu_total=2, icu_occupied=2)
print(f"Mesh online. ICU capacity: total={capacity.icu_total}, occupied={capacity.icu_occupied}, free={capacity.icu_free}")


## Troponin 0/1h classifier (default: Abbott hs‑cTnI)  
Units **ng/L**. If your LIS reports **ng/mL**, multiply by 1000 before passing values.

In [ ]:

ASSAYS = {
    "Abbott_Architect_hs_cTnI": {
        "units": "ng/L",
        "rule_in":  {"abs_0h": 64.0, "delta_1h": 6.0},
        "rule_out": {"single_0h": 2.0, "band_upper": 5.0, "delta_1h": 2.0},
        "imprecision_note": "Caution near LoD; Δ<2 ng/L relies on tight QC."
    },
    "Roche_hs_cTnT": {
        "units": "ng/L",
        "rule_in":  {"abs_0h": 52.0, "delta_1h": 5.0},
        "rule_out": {"single_0h": 5.0, "band_upper": 12.0, "delta_1h": 3.0}
    }
}

def classify_troponin_0_1h(assay_key: str, t0: float, t1: float):
    spec = ASSAYS[assay_key]
    delta = t1 - t0
    ri_abs  = spec["rule_in"]["abs_0h"]
    ri_d1   = spec["rule_in"]["delta_1h"]
    ro_single = spec["rule_out"]["single_0h"]
    ro_band_up = spec["rule_out"]["band_upper"]
    ro_d1   = spec["rule_out"]["delta_1h"]
    if t0 >= ri_abs or delta >= ri_d1:
        return "rule_in", {"t0": t0, "t1": t1, "delta": delta, "trigger": "abs_0h" if t0 >= ri_abs else "delta_1h"}
    if t0 < ro_single:
        return "rule_out", {"t0": t0, "t1": t1, "delta": delta, "trigger": "single_0h"}
    if t0 < ro_band_up and delta < ro_d1:
        return "rule_out", {"t0": t0, "t1": t1, "delta": delta, "trigger": "band+delta"}
    return "observe", {"t0": t0, "t1": t1, "delta": delta, "trigger": "observe_zone"}

DEFAULT_ASSAY = "Abbott_Architect_hs_cTnI"
print("[Troponin classifier ready] Default:", DEFAULT_ASSAY)


In [ ]:

# --- ACS candidate logic + ECG traffic light (uses troponin class) ---
acs_policy = {
    "equivalent_symptoms": {"dyspnea","epigastric_pain","jaw_arm_back_pain",
                            "nausea_vomiting","diaphoresis","syncope_unexplained","fatigue_unexplained"},
    "risk_modifiers": {"age_ge_65","female","diabetes","ckd","known_cad"},
    "vital_red_flags": {"map_lt_65","spo2_le_92","hr_gt_130","hr_lt_40"},
    "treat_as_acs_if": {"min_equiv_symptoms": 1, "or_if_equiv_0_then_risk_and_redflag": True}
}

def compute_acs_candidate(triage_flags: dict, vitals: dict, risk_flags: dict) -> bool:
    eq, rf, red = acs_policy["equivalent_symptoms"], acs_policy["risk_modifiers"], acs_policy["vital_red_flags"]
    n_equiv = sum(bool(triage_flags.get(k)) for k in eq)
    has_risk = any(bool(risk_flags.get(k)) for k in rf)
    map_lt_65   = vitals.get("MAP", 80) < 65
    spo2_le_92  = vitals.get("SpO2", 97) <= 92
    hr          = vitals.get("HR", 80)
    hr_flag     = (hr > 130 or hr < 40)
    has_red     = map_lt_65 or spo2_le_92 or hr_flag
    if n_equiv >= acs_policy["treat_as_acs_if"]["min_equiv_symptoms"]:
        return True
    if acs_policy["treat_as_acs_if"]["or_if_equiv_0_then_risk_and_redflag"]:
        return has_risk and has_red
    return False

def get_prior_ecg_summary(encounter_id):
    return {"when": "2025-07-29T14:10Z", "metrics": {"QTc": 430}, "st_mm_by_lead": {"V2": 0.0, "V3": 0.0}, "phenotype":"NORMAL"}

def compute_ecg_deltas(current, prior):
    if not prior:
        return {"delta_st_mm_max": 0.0, "reciprocal_st_mm_max": 0.0, "unchanged_vs_prior": None}
    d_v2 = current.get("st_mm_by_lead",{}).get("V2",0.0) - prior.get("st_mm_by_lead",{}).get("V2",0.0)
    d_v3 = current.get("st_mm_by_lead",{}).get("V3",0.0) - prior.get("st_mm_by_lead",{}).get("V3",0.0)
    delta_st = max(d_v2, d_v3)
    recip = current.get("reciprocal_st_mm_max", 0.0)
    unchanged = (abs(delta_st) < 0.5 and recip < 0.5)
    return {"delta_st_mm_max": float(round(delta_st,2)),
            "reciprocal_st_mm_max": float(round(recip,2)),
            "unchanged_vs_prior": unchanged}

def is_abnormal_vitals(vitals):
    return (vitals.get("MAP", 80) < 65) or (vitals.get("SpO2", 97) < 92) or (vitals.get("HR", 80) > 130) or (vitals.get("HR",80) < 40)

def ecg_traffic_light(ai, compare, ctx, troponin_class, cfg):
    if ai["metrics"].get("vt_vf"): return "RED", ["VT/VF"], "Page attending"
    if ai["metrics"].get("chb") and ai["metrics"].get("brady_hypotension"): return "RED", ["Complete heart block + hypotension/LOC"], "Page attending"
    if ctx.get("shock"):
        if ai["phenotype"] == "HyperK" or (ctx.get("k_value") is not None and ctx["k_value"] >= 6.5):
            return "RED", ["HyperK pattern + shock/↑K"], "Page attending"
    meets_static = (ai["severity"] == "CRITICAL")
    dynamic = (compare.get("delta_st_mm_max", 0.0) >= cfg["delta_st_mm"] or
               compare.get("reciprocal_st_mm_max", 0.0) >= cfg["recip_mm"])
    context_ok = (ctx.get("acs_candidate") or ctx.get("abnormal_vitals"))
    conf_ok = (ai.get("confidence", 0.0) >= cfg["min_conf"])
    if troponin_class == "rule_in":
        return "RED", ["Troponin rule-in per assay 0/1h"], "Page attending"
    confounder = ai.get("confounder", "none")
    if meets_static and dynamic and context_ok and conf_ok:
        if confounder in {"lbbb","paced","lvh_strain","early_repol","pericarditis"}:
            if troponin_class == "rule_in":
                return "RED", ["Occlusion + dynamics + context + Δtroponin"], "Page attending"
        else:
            return "RED", ["Occlusion + dynamics + context"], "Page attending"
    if ai["metrics"].get("HR", 0) > 150 and ai["phenotype"] in {"AF_RVR","Tachy"}: return "YELLOW", ["AF with RVR >150"], "Show resident"
    if ai["metrics"].get("QTc", 0) >= 500: return "YELLOW", ["QTc ≥ 500 ms"], "Show resident"
    if ai["phenotype"] in {"Wellens","Brugada","PacerFailure","NewLBBB"}: return "YELLOW", [ai["phenotype"]], "Show resident"
    if meets_static or ai["severity"] == "URGENT": return "YELLOW", ["Abnormal ECG—needs review"], "Show resident"
    if compare.get("unchanged_vs_prior") is False: return "YELLOW", ["New changes vs prior but not RED"], "Show resident"
    if troponin_class == "observe": return "YELLOW", ["Troponin observe-zone; needs repeat"], "Show resident"
    return "GREEN", ["Normal/benign; unchanged; low risk"], "Log only"

def triage_standing_orders(encounter_id, vitals, context_flags, triage_flags=None, risk_flags=None):
    triage_flags = triage_flags or {}; risk_flags = risk_flags or {}
    acs_cand = compute_acs_candidate(triage_flags, vitals, risk_flags)
    ctx = {
        "identity_bound": True,
        "two_identifiers_checked": True,
        "barcode_label_printed": True,
        "airway_compromise": context_flags.get("A", False),
        "severe_resp_distress": context_flags.get("B", False),
        "spo2_room_air": vitals.get("SpO2", 97),
        "shock_or_map_lt_65": (vitals.get("MAP", 80) < 65) or context_flags.get("C_critical", False),
        "icu_free_beds": capacity.icu_free,
        "acs_candidate": acs_cand
    }
    on_capacity_change(encounter_id, capacity.icu_free)
    policy_emit(encounter_id, "proposal.order.ecg", {"priority":"STAT"}, ctx, raw_emit)
    policy_emit(encounter_id, "proposal.order.labs", {"panel_id":"ed_big_panel","priority":"STAT"}, ctx, raw_emit)
    policy_emit(encounter_id, "proposal.request.vbga", {"site":"venous"}, ctx, raw_emit)
    if (ctx["airway_compromise"] or ctx["severe_resp_distress"] or ctx["spo2_room_air"] <= 92 or ctx["shock_or_map_lt_65"]):
        policy_emit(encounter_id, "proposal.request.abga", {"site":"arterial"}, ctx, raw_emit)
    return ctx

def on_ecg_and_troponin(encounter_id, ecg_ai, vitals, t0, t1, assay_key=None, acs_candidate_hint=None):
    assay_key = assay_key or DEFAULT_ASSAY
    prior = get_prior_ecg_summary(encounter_id)
    compare = compute_ecg_deltas(ecg_ai, prior)
    acs_cand = acs_candidate_hint if acs_candidate_hint is not None else is_abnormal_vitals(vitals)
    ctx = {"acs_candidate": acs_cand, "abnormal_vitals": is_abnormal_vitals(vitals),
           "k_value": None, "shock": vitals.get("MAP",80) < 65}
    policy_emit(encounter_id, "proposal.ecg.interpret",
                {"ecg_id": ecg_ai.get("ecg_id","ecg0"), "source": ecg_ai.get("source","triage")},
                {"icu_free_beds": capacity.icu_free}, raw_emit)
    troponin_class, meta = classify_troponin_0_1h(assay_key, t0, t1)
    cfg = {"min_conf": 0.85, "delta_st_mm": 1.0, "recip_mm": 0.5}
    color, why, action = ecg_traffic_light(ecg_ai, compare, ctx, troponin_class, cfg)
    log_event("ECG  ", color, "; ".join(why) + f" | assay:{assay_key} troponin:{troponin_class} ({meta})")
    if color == "RED":
        policy_emit(encounter_id, "proposal.ecg.alert",
                    {"severity":"CRITICAL","phenotype":ecg_ai["phenotype"],"confidence":ecg_ai["confidence"]},
                    {"icu_free_beds": capacity.icu_free}, raw_emit)
        return color
    if ctx["acs_candidate"] and troponin_class != "rule_in":
        minutes = 60  # stay on 0/1h for demo; add 0/2h later if needed
        policy_emit(encounter_id, "proposal.ecg.repeat", {"minutes": minutes}, {"icu_free_beds": capacity.icu_free}, raw_emit)
        policy_emit(encounter_id, "proposal.request.troponin_repeat",
                    {"minutes": minutes, "assay": "hsTnI" if "cTnI" in assay_key else "hsTnT"},
                    {"icu_free_beds": capacity.icu_free}, raw_emit)
    if color == "YELLOW":
        policy_emit(encounter_id, "proposal.ecg.alert",
                    {"severity":"URGENT","phenotype":ecg_ai["phenotype"],"confidence":ecg_ai["confidence"]},
                    {"icu_free_beds": capacity.icu_free}, raw_emit)
    return color

print("[ECG/ACS path ready]")


In [ ]:

# --- SOP Pack Loader (unified) ---
PACKS = {}
def try_load(alias, path):
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            PACKS[alias] = yaml.safe_load(f)
        print(f"[loaded] {alias}: {path} (modules: {list(PACKS[alias].get('modules',{}).keys())})")
    else:
        print(f"[skip] {alias}: {path} not found")

try_load("jena", "/mnt/data/jena_sop_policy.yaml")
try_load("notaufnahme", "/mnt/data/sop_notaufnahme_policy.yaml")
try_load("taschen", "/mnt/data/taschenatlas_policy.yaml")
try_load("overrides", "/mnt/data/erc_awmf_overrides.yaml")
try_load("cap", "/mnt/data/cap_s3_2021_policy.yaml")
try_load("rush", "/mnt/data/rush_acils_policy.yaml")
try_load("geri", "/mnt/data/geriatrics_overlay.yaml")
try_load("neuro", "/mnt/data/neurology_ed_sop_heidelberg_2017.yaml")
try_load("psy",   "/mnt/data/notfallpsychiatrie_s2k_2019_policy.yaml")
try_load("sys",   "/mnt/data/system_overlay_notfallversorgung.yaml")

def sop_actions_for(alias: str, module_key: str):
    return PACKS[alias]["modules"][module_key]["actions"]

def apply_pack(encounter_id: str, alias: str, module_key: str, ctx: dict):
    actions = sop_actions_for(alias, module_key)
    for item in actions:
        (k, v), = item.items()
        if alias == "overrides":
            policy_emit(encounter_id, "proposal.page_team",
                        {"team":"physician","priority":"STAT","reason":f"{module_key}:{k} -> {v}"},
                        ctx, raw_emit); continue
        if k == "order_labs":
            policy_emit(encounter_id, "proposal.order.labs", {"panel_id": v.get("panel_id","ed_big_panel"), "priority": v.get("priority","STAT")}, ctx, raw_emit)
        elif k == "order_ecg":
            policy_emit(encounter_id, "proposal.order.ecg", {"priority": v.get("priority","STAT")}, ctx, raw_emit)
        elif k in {"ecg_repeat_batched","ecg_repeat"}:
            policy_emit(encounter_id, "proposal.ecg.repeat", {"minutes": v.get("minutes",15)}, ctx, raw_emit)
        elif k == "troponin_scheme":
            mins = 60 if str(v.get("scheme","0/1h")).startswith("0/1") else 120
            policy_emit(encounter_id, "proposal.request.troponin_repeat", {"minutes": mins, "assay": v.get("assay","hsTnI")}, ctx, raw_emit)
        elif k == "page_team":
            policy_emit(encounter_id, "proposal.page_team", {"team": v["team"], "priority": v.get("priority","URGENT"), "reason": v.get("reason","SOP")}, ctx, raw_emit)
        elif k in {"d_dimer"}:
            policy_emit(encounter_id, "proposal.order.labs", {"panel_id":"d_dimer","priority":"STAT"}, ctx, raw_emit)
        elif k in {"order.blood_cultures","order.urine_antigen","order.sputum"}:
            policy_emit(encounter_id, f"proposal.{k}", v, ctx, raw_emit)
        elif k in {"cap.score.crb65","cap.abx.duration"}:
            policy_emit(encounter_id, f"proposal.{k}", v, ctx, raw_emit)
        elif k == "imaging":
            policy_emit(encounter_id, "proposal.page_team", {"team":"radiology","priority":"STAT","reason":f"Imaging per SOP: {v}"}, ctx, raw_emit)
        else:
            policy_emit(encounter_id, "proposal.page_team", {"team":"physician","priority":"URGENT","reason":f"{alias}:{module_key}:{k} -> {v}"}, ctx, raw_emit)
    print(f"[SOP applied] {alias}:{module_key}")


In [ ]:

def demo_acs_default_abbott():
    print("\n=== DEMO: ACS (default Abbott hs‑cTnI) — rule‑out band ===")
    enc = "ED-ACS-ABBOTT-DEF-01"
    vitals = {"HR":96,"MAP":76,"SpO2":95}
    triage_flags = {"dyspnea":True}
    risk_flags = {"age_ge_65":True}
    ctx = triage_standing_orders(enc, vitals, {"A":False,"B":False,"C_critical":False}, triage_flags, risk_flags)
    ecg_ai = {"ecg_id":"ecgA","source":"triage","phenotype":"NON_DIAGNOSTIC","severity":"URGENT",
              "confidence":0.9,"confounder":"none","st_mm_by_lead":{"V2":0.3,"V3":0.4},"metrics":{"HR":96,"QTc":440}}
    t0, t1 = 3.0, 4.0  # Δ1h = 1 (<2) within 2–5 ng/L band -> rule-out
    on_ecg_and_troponin(enc, ecg_ai, vitals, t0, t1, None, acs_candidate_hint=ctx["acs_candidate"])

def demo_acs_abbott_rulein_delta():
    print("\n=== DEMO: ACS (Abbott hs‑cTnI) — rule‑in by Δ1h ===")
    enc = "ED-ACS-ABBOTT-RI-01"
    vitals = {"HR":88,"MAP":80,"SpO2":97}
    triage_flags = {"dyspnea":True}
    risk_flags = {"age_ge_65":True}
    ctx = triage_standing_orders(enc, vitals, {"A":False,"B":False,"C_critical":False}, triage_flags, risk_flags)
    ecg_ai = {"ecg_id":"ecgB","source":"triage","phenotype":"NON_DIAGNOSTIC","severity":"URGENT",
              "confidence":0.92,"confounder":"none","st_mm_by_lead":{"V2":0.2,"V3":0.3},"metrics":{"HR":88,"QTc":438}}
    t0, t1 = 18.0, 25.0  # Δ1h = 7 >= 6 -> rule-in
    on_ecg_and_troponin(enc, ecg_ai, vitals, t0, t1, "Abbott_Architect_hs_cTnI", acs_candidate_hint=ctx["acs_candidate"])

def demo_pe_cap_psy():
    print("\n=== DEMO: PE SOP + CAP hooks + Psych agitation ===")
    ctx = {"icu_free_beds": capacity.icu_free}
    if "notaufnahme" in PACKS:
        apply_pack("ED-PE-01", "notaufnahme", "lungenarterienembolie", ctx)
    if "cap" in PACKS:
        apply_pack("ED-CAP-01", "cap", "cap_triage_and_severity", ctx)
        apply_pack("ED-CAP-01", "cap", "cap_microbiology_inpatient", ctx)
    if "psy" in PACKS:
        apply_pack("ED-PSY-01", "psy", "agitation_deescalation", ctx)

demo_acs_default_abbott()
demo_acs_abbott_rulein_delta()
demo_pe_cap_psy()
print("\n— Flow Bundle v1.2 demo complete —")
